In [1]:
from SiNAPSE.core import Neuron, Recording, Database
from _neurons import Response, LatencyCalculator

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from win32cryptcon import szOID_PKCS_1

In [2]:
RECORDINGS_PATH = r'N:\Data\RhythmPerception\Neural Recordings'
DB_PATH = r'C:\Users\tmerri03\Desktop\Temp Neural Files\awake_recordings.db'
STIM_LIB_PATH = r'R:\Data\tyler\Recordings\Stim\Stimuli Library'
db = Database(DB_PATH, STIM_LIB_PATH)

In [3]:
rec_path = r'N:\Data\RhythmPerception\Neural Recordings\o2o55\o3o55 Recording #2 (77-H2, Field L, 1.4AP 1.2ML-R 1708um)'
recording = Path(rec_path).name
bird = recording.split(' ')[0]
rec = Recording(rec_path, samplerate=30000, db=db)

unit = 217

In [4]:
N = Neuron(recording, unit, rec=rec, db=db)
stims=N.load
N.plot_waveform()

R=Response(STIM_LIB_PATH, RECORDINGS_PATH, recording, db=db)
baselines=R.baseline_fr(return_raster=True)

# ------------------------------------------------------------------ #
#  Plot baseline activity windows
# ------------------------------------------------------------------ #
fig, ax = plt.subplots(figsize=(10, 5))

raster = baselines[unit]['raster']
duration = baselines[unit]['duration']
nstimuli = baselines[unit]['nstimuli']

for trial_idx, spike_times in enumerate(raster):
    spike_times = np.asarray(spike_times)
    ax.vlines(spike_times, trial_idx, trial_idx + 0.8, color='black', linewidth=0.8)

# mark where each stitched stimulus-baseline segment starts
for s in range(1, nstimuli):
    ax.axvline(s * duration, color='red', linestyle='--', linewidth=0.5, alpha=0.5)

ax.set_xlabel(f'Time (s) — {nstimuli} stitched {duration}s baseline segments')
ax.set_ylabel('Trial index')
ax.set_title(f"Unit {unit} — baseline spontaneous activity\n"
             f"mean={baselines[unit]['mean']:.2f} Hz, std={baselines[unit]['std']:.2f} Hz")
ax.set_ylim(-0.5, len(raster))
ax.set_xlim(0, nstimuli * duration)
plt.show()

In [10]:
stim_subset = ['exp1_ZF A_20db_180ms_8b_1xomit_8b_silence',
               'exp1_ZF A_20db_220ms_8b_1xomit_8b_silence'
               ]
# N.plot(stimuli_to_raster=stim_subset, baseline=True, padding=0)

In [11]:
import numpy as np
import matplotlib.pyplot as plt

class OmissionAnalysis:
    def __init__(self, N, stimulus, tempo_ms, n_beats_before, n_omissions, n_beats_after, padding=1):
        """
        N              : your neuron object
        stimulus       : stimulus identifier
        tempo_ms       : beat period in ms (e.g. 180)
        n_beats_pre    : number of beats before the sequence to analyze
        n_beats_before : number of beats before the omission
        n_omissions    : number of consecutive omitted beats
        n_beats_after  : number of beats after the omission
        padding        : padding passed to N.raster (1 = spike times relative to padding onset)
        """

        self.N = N
        self.stimulus = stimulus
        self.tempo_ms = tempo_ms
        self.tempo_s  = tempo_ms / 1000
        self.n_beats_before = n_beats_before
        self.n_beats_after = n_beats_after
        self.n_omissions = n_omissions
        self.padding = padding

        self.trial_spikes = None
        self.duration = None
        self.beat_times = None       # in seconds, relative to trial onset (padding=0 frame)
        self.omission_times = None   # expected beat times during omission window


    # ------------------------------------------------------------------ #
    #  Step 1 — load spikes
    # ------------------------------------------------------------------ #
    def get_spike_times(self):
        _, self.trial_spikes, self.duration = self.N.raster(
            stimulus=self.stimulus, baseline=False,
            ax=None, plot=False, separate_trials=True,
            padding=self.padding
        )

        return self

    # ------------------------------------------------------------------ #
    #  Step 2 — compute beat times  ← YOU FILL THIS IN
    # ------------------------------------------------------------------ #
    def compute_beat_times(self, silence_duration_s, sound_duration_s, gap_duration_s, n_decay_windows=None, n_pre_windows=None):
        """
        n_decay_windows : number of extra beat positions after post_times to track decay.
                          Defaults to filling the remaining trial duration.
        """
        first_beat = silence_duration_s
        ibi = self.tempo_s

        self.pre_times = np.array([first_beat - ibi - i * ibi
                                  for i in range(n_pre_windows)])

        self.beat_times = np.array([first_beat + i * ibi
                                    for i in range(self.n_beats_before)])

        self.omission_times = np.array([first_beat + (self.n_beats_before + i) * ibi
                                        for i in range(self.n_omissions)])

        self.post_times = np.array([first_beat + (self.n_beats_before + self.n_omissions + i) * ibi
                                    for i in range(self.n_beats_after)])

        # decay windows: continue beat sequence after post_times until end of trial
        post_end_idx = self.n_beats_before + self.n_omissions + self.n_beats_after
        if n_decay_windows is None:
            n_decay_windows = int((self.duration - first_beat) / ibi) - post_end_idx
            n_decay_windows = max(n_decay_windows, 0)

        self.decay_times = np.array([first_beat + (post_end_idx + i) * ibi
                                     for i in range(n_decay_windows)])
        self.decay_times = self.decay_times[self.decay_times < self.duration+self.padding]

        return self

    # ------------------------------------------------------------------ #
    #  Step 2.5 — Calculate latencies
    # ------------------------------------------------------------------ #
    def compute_sound_latencies(self, latency_control_sound, search_window=0.100,
                             n_bootstrap=1000, percentile=99, plot=True):
        """
        Computes latency for `latency_control_sound` using bootstrap thresholding
        from this unit's own baseline spike trains.
        """
        print(search_window)
        baseline_trials, ntrials, nstimuli = self.N.collect_baseline(duration=search_window)

        calc = LatencyCalculator(self.N)
        calc.fit_baseline(
            baseline_trials,
            search_window=search_window,
            n_bootstrap=n_bootstrap,
            percentile=percentile,
        )

        _, spks, _ = self.N.raster(
            stimulus=latency_control_sound, baseline=False,
            ax=None, plot=False, separate_trials=False,
            padding=0
        )

        latency = calc.calculate_latency(spks, search_window=search_window)

        if plot:
            calc.plot(spks, search_window=search_window, latency=latency)

        self.latency_calc = calc  # keep around for inspection/debugging
        self.latency = latency

        return self

    # ------------------------------------------------------------------ #
    #  Step 3 — phase locking per beat position
    # ------------------------------------------------------------------ #
    from scipy.stats import circmean, circstd

    def compute_phase_locking(self, window_s=None, latency_s=20/1000):
        if self.beat_times is None:
            raise RuntimeError("Call compute_beat_times() first.")
        if window_s is None:
            window_s = self.tempo_s

        all_times = np.concatenate([self.pre_times, self.beat_times,
                                     self.omission_times, self.post_times,
                                     self.decay_times])
        labels = (['pre']      * len(self.pre_times) +
                  ['beat']     * len(self.beat_times) +
                  ['omission'] * len(self.omission_times) +
                  ['beat']     * len(self.post_times) +
                  ['decay']    * len(self.decay_times))

        self.phase_locking = []

        for beat_t, label in zip(all_times, labels):
            all_phases = []
            for spikes in self.trial_spikes:
                spikes = np.array(spikes)
                near = spikes[
                    (spikes >= beat_t + latency_s) &
                    (spikes <  beat_t + latency_s + window_s)
                ]
                phases = ((near - beat_t) / self.tempo_s) * 2 * np.pi
                all_phases.extend(phases.tolist())

            phases = np.array(all_phases)

            if len(phases) > 0:
                z = np.mean(np.exp(1j * phases))
                vs         = np.abs(z)
                mean_phase = np.angle(z)                          # radians, -π to π
                mean_phase_ms = (mean_phase / (2 * np.pi)) * self.tempo_ms  # ms into beat cycle
            else:
                vs = np.nan
                mean_phase = np.nan
                mean_phase_ms = np.nan

            self.phase_locking.append({
                'time':          beat_t,
                'label':         label,
                'vector_strength': vs,
                'mean_phase':    mean_phase,       # radians
                'mean_phase_ms': mean_phase_ms,    # ms — most interpretable
                'n_spikes':      len(phases),
                'phases':        phases,           # keep raw phases for comparison test
            })

        self.latency_s = latency_s
        self._compare_beat_vs_omission()           # run comparison automatically
        return self


    def _compare_beat_vs_omission(self):
        """
        Compares mean phase of beat responses vs omission responses.
        Stores the angular difference and a bootstrap p-value.
        """
        beat_phases = np.concatenate([
            d['phases'] for d in self.phase_locking if d['label'] == 'beat'
        ])
        omission_phases = np.concatenate([
            d['phases'] for d in self.phase_locking if d['label'] == 'omission'
        ])

        if len(beat_phases) == 0 or len(omission_phases) == 0:
            self.phase_comparison = None
            return

        beat_mean     = np.angle(np.mean(np.exp(1j * beat_phases)))
        omission_mean = np.angle(np.mean(np.exp(1j * omission_phases)))

        # angular difference, wrapped to [-π, π]
        diff = np.angle(np.exp(1j * (omission_mean - beat_mean)))
        diff_ms = (diff / (2 * np.pi)) * self.tempo_ms

        # bootstrap p-value: how often does random shuffling produce
        # a difference this small or smaller?
        all_phases = np.concatenate([beat_phases, omission_phases])
        n_beat = len(beat_phases)
        n_boot = 1000

        null_diffs = np.zeros(n_boot)
        for i in range(n_boot):
            shuffled = np.random.permutation(all_phases)
            b = np.angle(np.mean(np.exp(1j * shuffled[:n_beat])))
            o = np.angle(np.mean(np.exp(1j * shuffled[n_beat:])))
            null_diffs[i] = np.abs(np.angle(np.exp(1j * (o - b))))

        # p = fraction of null differences *smaller* than observed
        # (small difference = similar phases = what you're hoping for)
        observed_diff = np.abs(diff)
        p_value = np.mean(null_diffs <= observed_diff)

        self.phase_comparison = {
            'beat_mean_phase_ms':     (beat_mean / (2 * np.pi)) * self.tempo_ms,
            'omission_mean_phase_ms': (omission_mean / (2 * np.pi)) * self.tempo_ms,
            'difference_ms':          diff_ms,
            'p_value':                p_value,   # small p = phases are more similar than chance
        }

        print(f"\n--- Phase comparison: beat vs omission ---")
        print(f"  Beat mean phase:     {self.phase_comparison['beat_mean_phase_ms']:+.1f} ms into beat cycle")
        print(f"  Omission mean phase: {self.phase_comparison['omission_mean_phase_ms']:+.1f} ms into beat cycle")
        print(f"  Difference:          {diff_ms:+.1f} ms")
        print(f"  Bootstrap p-value:   {p_value:.3f}  (small = phases are similar)")

    # ------------------------------------------------------------------ #
    #  Step 4 — plot
    # ------------------------------------------------------------------ #
    def plot(self):
        if not hasattr(self, 'phase_locking'):
            raise RuntimeError("Call compute_phase_locking() first.")

        times  = np.array([d['time'] for d in self.phase_locking])
        vs     = np.array([d['vector_strength'] for d in self.phase_locking])
        labels = [d['label'] for d in self.phase_locking]

        colors = {'beat': 'steelblue', 'omission': 'tomato', 'decay': 'gray', 'pre': 'gray'}
        point_colors = [colors[l] for l in labels]

        # ------------------------------------------------------------------ #
        #  Compute mean phase from beat windows -> predicted spike time offset
        # ------------------------------------------------------------------ #
        beat_phases = np.concatenate([
            d['phases'] for d in self.phase_locking if d['label'] == 'beat'
        ])
        if len(beat_phases) > 0:
            mean_beat_phase = np.angle(np.mean(np.exp(1j * beat_phases)))
            mean_beat_phase = mean_beat_phase % (2 * np.pi)          # <-- remap negative wrap to [0, 2π)
            mean_beat_offset_s = (mean_beat_phase / (2 * np.pi)) * self.tempo_s
        else:
            mean_beat_offset_s = None

        fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

        # ------------------------------------------------------------------ #
        #  Top: raster
        # ------------------------------------------------------------------ #
        for i, spikes in enumerate(self.trial_spikes):
            axes[0].scatter(np.array(spikes) * 1000,
                            np.full(len(spikes), i),
                            s=1, c='steelblue', alpha=0.5)

        for t in self.pre_times:
            axes[0].axvline(t * 1000, color='gray', lw=0.5, alpha=1, ls=':')
        for t in self.beat_times:
            axes[0].axvline(t * 1000, color='steelblue', lw=0.5, alpha=1)
        for t in self.omission_times:
            axes[0].axvline(t * 1000, color='tomato', lw=1, ls='--', alpha=1)
        for t in self.post_times:
            axes[0].axvline(t * 1000, color='steelblue', lw=0.5, alpha=1)
        for t in self.decay_times:
            axes[0].axvline(t * 1000, color='gray', lw=0.5, alpha=1, ls=':')

        # predicted spike time in each omission window
        if mean_beat_offset_s is not None:
            for t in self.omission_times:
                predicted_ms = (t + mean_beat_offset_s) * 1000
                axes[0].axvline(predicted_ms, color='tomato', lw=1.5,
                                ls=':', alpha=0.9,
                                label='predicted (beat mean phase)' if t == self.omission_times[0] else None)

        axes[0].set_ylabel('Trial')
        axes[0].set_title(self.stimulus)
        axes[0].legend(loc='upper right', fontsize=7)

        # ------------------------------------------------------------------ #
        #  Bottom: vector strength
        # ------------------------------------------------------------------ #
        axes[1].scatter((times + self.tempo_s / 2) * 1000, vs,
                        c=point_colors, s=40, zorder=3)
        axes[1].axhline(0, color='gray', lw=0.5)

        for t in self.pre_times:
            axes[1].axvspan(
                (t + self.latency_s) * 1000,
                (t + self.tempo_s + self.latency_s) * 1000,
                alpha=0.07, color='gray'
            )
        for t in self.omission_times:
            axes[1].axvspan(
                (t + self.latency_s) * 1000,
                (t + self.tempo_s + self.latency_s) * 1000,
                alpha=0.1, color='tomato'
            )
            # predicted spike time in VS panel too
            if mean_beat_offset_s is not None:
                predicted_ms = (t + self.latency_s + mean_beat_offset_s) * 1000
                axes[1].axvline(predicted_ms, color='tomato', lw=1.5, ls=':',  alpha=0.9)

        for t in self.decay_times:
            axes[1].axvspan(
                (t + self.latency_s) * 1000,
                (t + self.tempo_s + self.latency_s) * 1000,
                alpha=0.07, color='gray'
            )

        axes[1].set_xlabel('Time (ms)')
        axes[1].set_ylabel('Vector strength')
        axes[1].set_title('Phase locking per beat position  (blue = beat, red = omission, dotted = predicted)')
        axes[1].set_ylim(0, 1)

        plt.tight_layout()
        plt.show()

In [12]:
# ------------------------------------------------------------------ #
#  Step 5 — polar (rose) plot of phase locking
# ------------------------------------------------------------------ #
def _plot_polar(self, windows=('beat', 'omission'), bins=24, ax=None, show=True):
    """
    Polar plot of spike-phase distribution relative to the beat cycle.

    - Angle       : phase within the beat cycle (0 = beat onset, going clockwise
                    toward the next beat), i.e. "timing of the response".
    - Bar height  : fraction of that window's spikes falling in each phase bin
                    ("tightness" of firing -- tall/narrow bars = tightly locked,
                    low/spread-out bars = dispersed/jittery firing). Heights are
                    normalized by spike count so tightness is comparable across
                    stimuli, windows, and units regardless of firing rate.
    - Arrow       : mean resultant (circular-mean) vector for that window.
                    Direction = mean phase, length = vector strength (0-1),
                    on the same radial scale as the bars.

    windows : which phase_locking labels to overlay, e.g. ('beat','omission')
    bins    : number of phase bins for the rose histogram
    ax      : existing polar Axes to draw into (creates one if None)
    show    : call plt.show() if a new figure was created
    """
    if not hasattr(self, 'phase_locking'):
        raise RuntimeError("Call compute_phase_locking() first.")

    colors = {'beat': 'steelblue', 'omission': 'tomato', 'decay': 'gray', 'pre': 'gray'}

    created_fig = False
    if ax is None:
        fig = plt.figure(figsize=(6, 6))
        ax = fig.add_subplot(111, projection='polar')
        created_fig = True

    ax.set_theta_zero_location('N')
    ax.set_theta_direction(-1)

    edges = np.linspace(0, 2 * np.pi, bins + 1)
    width = edges[1] - edges[0]
    centers = edges[:-1] + width / 2

    window_data = []
    rmax = 0.0
    for label in windows:
        phases = np.concatenate(
            [d['phases'] for d in self.phase_locking if d['label'] == label]
        ) if any(d['label'] == label for d in self.phase_locking) else np.array([])
        if len(phases) == 0:
            continue

        counts, _ = np.histogram(np.mod(phases, 2 * np.pi), bins=edges)
        density = counts / counts.sum()

        z = np.mean(np.exp(1j * phases))
        vs = np.abs(z)
        mean_phase = np.angle(z) % (2 * np.pi)

        window_data.append(dict(label=label, phases=phases, density=density,
                                 vs=vs, mean_phase=mean_phase))
        rmax = max(rmax, density.max(), vs)

    for wd in window_data:
        color = colors.get(wd['label'], 'gray')
        ax.bar(centers, wd['density'], width=width * 0.9, bottom=0,
               color=color, alpha=0.35, edgecolor=color, linewidth=0.5,
               label=f"{wd['label']} spikes (n={len(wd['phases'])})")
        ax.annotate('', xy=(wd['mean_phase'], wd['vs']), xytext=(0, 0),
                     arrowprops=dict(facecolor=color, edgecolor=color,
                                      width=2, headwidth=8, alpha=0.9))

    if rmax > 0:
        ax.set_ylim(0, rmax * 1.15)

    ax.set_title(f"{self.stimulus}\nphase distribution (bars) + mean vector (arrow)",
                 fontsize=10)
    ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=7)

    if created_fig and show:
        plt.tight_layout()
        plt.show()

    return ax


OmissionAnalysis.plot_polar = _plot_polar


In [13]:
import re, scipy, os

def parse_stimulus_string(s, raw_sounds_fp, stim_lib_fp):
    fs, data = scipy.io.wavfile.read(
        os.path.join(stim_lib_fp, f'{s}.wav')
    )
    total_duration = len(data)/fs

    s = s.replace(' ', '_').replace('2khz', '2_khz')

    # extract structured fields by pattern, not position
    tempo        = int(re.search(r'(\d+)ms', s).group(1))
    n_before     = int(re.search(r'(\d+)b_\d+xomit', s).group(1))
    n_omissions  = int(re.search(r'(\d+)xomit', s).group(1))
    n_after      = int(re.search(r'xomit_(\d+)b', s).group(1))

    # sound name is everything before the tempo token
    raw_sound = re.search(r'^(.+?)_\d+ms_', s).group(1)
    raw_sound = raw_sound.replace('2_khz', '2khz')

    fs, data = scipy.io.wavfile.read(
        os.path.join(raw_sounds_fp, f'{raw_sound}.wav')
    )
    raw_sound_duration = len(data) / fs
    gap_duration       = tempo/1000 - raw_sound_duration

    if total_duration > (n_before+n_omissions+n_after)*tempo/1000:
        omission_duration  = n_omissions * (tempo/1000) + gap_duration
    else:
        omission_duration = 0

    return {
        'tempo':              tempo,
        'n_beats_before':     n_before,
        'n_omissions':        n_omissions,
        'n_beats_after':      n_after,
        'raw_sound':          raw_sound,
        'raw_sound_duration': raw_sound_duration,
        'gap_duration':       gap_duration,
        'omission_duration':  omission_duration,
        'latency_control_sound': f'{raw_sound.replace('ZF_A_20db', 'ZF A_20db')}_180_onebeatcontrol_silence'
    }

stims = {}
for s in stim_subset:
    stims[s] = parse_stimulus_string(s,
                                     raw_sounds_fp = r'C:\Users\tmerri03\Desktop\RhythmStimuli\Awake Recs\Raw Files\regex',
                                     stim_lib_fp = r'R:\Data\tyler\Recordings\Stim\Stimuli Library')

In [14]:
import pprint
for stimulus in stims.keys():
    print('Analyzing stimulus:', stimulus)
    stim_data = stims.get(stimulus)
    pprint.pprint(stim_data)

    oa = OmissionAnalysis(N, stimulus=stimulus,
                          tempo_ms=stim_data['tempo'],
                          n_beats_before=stim_data['n_beats_before'],
                          n_omissions=stim_data['n_omissions'],
                          n_beats_after=stim_data['n_beats_after'])
    oa.get_spike_times()
    oa.compute_beat_times(silence_duration_s=stim_data['omission_duration'],
                          sound_duration_s=stim_data['raw_sound_duration'],
                          gap_duration_s=stim_data['raw_sound_duration'],
                          n_pre_windows=5,
                          n_decay_windows=5,)
    latency=oa.compute_sound_latencies(latency_control_sound=stim_data['latency_control_sound'])

    print(latency.latency)
    oa.compute_phase_locking()
    oa.plot()
    oa.plot_polar()  # timing (angle) + tightness (bar height/arrow length) in omission window

Analyzing stimulus: exp1_ZF A_20db_180ms_8b_1xomit_8b_silence
{'gap_duration': 0.09859410430839001,
 'latency_control_sound': 'exp1_ZF A_20db_180_onebeatcontrol_silence',
 'n_beats_after': 8,
 'n_beats_before': 8,
 'n_omissions': 1,
 'omission_duration': 0.27859410430839,
 'raw_sound': 'exp1_ZF_A_20db',
 'raw_sound_duration': 0.08140589569160998,
 'tempo': 180}
0.1
None

--- Phase comparison: beat vs omission ---
  Beat mean phase:     -8.5 ms into beat cycle
  Omission mean phase: +1.7 ms into beat cycle
  Difference:          +10.1 ms
  Bootstrap p-value:   0.998  (small = phases are similar)
Analyzing stimulus: exp1_ZF A_20db_220ms_8b_1xomit_8b_silence
{'gap_duration': 0.13859410430839003,
 'latency_control_sound': 'exp1_ZF A_20db_180_onebeatcontrol_silence',
 'n_beats_after': 8,
 'n_beats_before': 8,
 'n_omissions': 1,
 'omission_duration': 0.35859410430839,
 'raw_sound': 'exp1_ZF_A_20db',
 'raw_sound_duration': 0.08140589569160998,
 'tempo': 220}
0.1
None

--- Phase comparison: be